**За допомогою скрипта Python підключаємось до бази даних в Google BigQuery (DA dataset)**


In [1]:
!pip install --upgrade google-cloud-bigquery

In [2]:
from google.colab import auth
from google.cloud import bigquery
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.stats.proportion import proportions_ztest

In [3]:
auth.authenticate_user()

In [4]:
client = bigquery.Client(project="796019654077")

**На основі таблиць, наявних у базі даних, напишемо запит у Python щоб створити датасет**

In [5]:
query = """
with session_info as (
SELECT
s.date,
s.ga_session_id,
sp.country,
sp.device,
sp.continent,
sp.channel,
ab.test,
ab.test_group
from data-analytics-mate.DA.ab_test as ab
join `DA.session` as s
on ab.ga_session_id = s.ga_session_id
join `DA.session_params` as sp
on sp.ga_session_id = s.ga_session_id
),
session_with_orders as (
SELECT
session_info.date,
-- session_info.ga_session_id,
session_info.country,
session_info.device,
session_info.continent,
session_info.channel,
session_info.test,
session_info.test_group,
count(distinct o.ga_session_id) as session_with_orders
FROM `DA.order` as o
join session_info
on o.ga_session_id = session_info.ga_session_id
group by
session_info.date,
session_info.country,
session_info.device,
session_info.continent,
session_info.channel,
session_info.test,
session_info.test_group
),
event as (
SELECT
    session_info.date,
    -- session_info.ga_session_id,
    session_info.country,
    session_info.device,
    session_info.continent,
    session_info.channel,
    session_info.test,
    session_info.test_group,
    sp.event_name,
    count(sp.ga_session_id) as event_cnt
FROM `DA.event_params` as sp
join session_info
on sp.ga_session_id = session_info.ga_session_id
group by
    session_info.date,
    -- session_info.ga_session_id,
    session_info.country,
    session_info.device,
    session_info.continent,
    session_info.channel,
    session_info.test,
    session_info.test_group,
    sp.event_name
),
session as (
select
    session_info.date,
    session_info.country,
    session_info.device,
    session_info.continent,
    session_info.channel,
    session_info.test,
    session_info.test_group,
    count(distinct session_info.ga_session_id) as session_cnt
from session_info
group by
    session_info.date,
    session_info.country,
    session_info.device,
    session_info.continent,
    session_info.channel,
    session_info.test,
    session_info.test_group
),
account as (
select
    session_info.date,
    session_info.country,
    session_info.device,
    session_info.continent,
    session_info.channel,
    session_info.test,
    session_info.test_group,
    count(distinct acs.ga_session_id) as new_account_cnt
from `DA.account_session` as acs
join session_info
on acs.ga_session_id = session_info.ga_session_id
group by
    session_info.date,
    session_info.country,
    session_info.device,
    session_info.continent,
    session_info.channel,
    session_info.test,
    session_info.test_group
)
select
    session_with_orders.date,
    session_with_orders.country,
    session_with_orders.device,
    session_with_orders.continent,
    session_with_orders.channel,
    session_with_orders.test,
    session_with_orders.test_group,
    'session_with_orders' as event_name,
    session_with_orders.session_with_orders as value
from
session_with_orders
union all
select
    event.date,
    event.country,
    event.device,
    event.continent,
    event.channel,
    event.test,
    event.test_group,
    event_name,
    event.event_cnt as value
from event
union all
select
    session.date,
    session.country,
    session.device,
    session.continent,
    session.channel,
    session.test,
    session.test_group,
    'session' as event_name,
    session.session_cnt as value
from session
union all
select
    account.date,
    account.country,
    account.device,
    account.continent,
    account.channel,
    account.test,
    account.test_group,
    'new account' as event_name,
    account.new_account_cnt as value
from account
"""


query_job = client.query(query)
results = query_job.result()
df = results.to_dataframe(create_bqstorage_client=False)
df

,date,country,device,continent,channel,test,test_group,event_name,value
0,2020-11-03,Qatar,mobile,Asia,Organic Search,2,2,new account,1
1,2020-11-03,Ecuador,mobile,Americas,Direct,2,2,new account,1
2,2020-11-12,New Zealand,mobile,Oceania,Undefined,2,2,new account,1
3,2020-11-12,Bulgaria,mobile,Europe,Paid Search,2,2,new account,1
4,2020-11-15,Bulgaria,desktop,Europe,Social Search,2,2,new account,1
...,...,...,...,...,...,...,...,...,...
800991,2020-12-22,Vietnam,mobile,Asia,Organic Search,4,1,first_visit,1
800992,2020-11-25,Vietnam,mobile,Asia,Paid Search,3,2,user_engagement,1
800993,2020-11-30,Vietnam,desktop,Asia,Organic Search,3,2,user_engagement,1
800994,2021-01-02,Vietnam,mobile,Asia,Organic Search,4,2,scroll,1


**Перевіряємо тип отриманих даних та нульові значення**

In [6]:
print(df.info())
print(df.isnull().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 800996 entries, 0 to 800995
Data columns (total 9 columns):
 #   Column      Non-Null Count   Dtype 
---  ------      --------------   ----- 
 0   date        800996 non-null  dbdate
 1   country     800996 non-null  object
 2   device      800996 non-null  object
 3   continent   800996 non-null  object
 4   channel     800996 non-null  object
 5   test        800996 non-null  Int64 
 6   test_group  800996 non-null  Int64 
 7   event_name  800996 non-null  object
 8   value       800996 non-null  Int64 
dtypes: Int64(3), dbdate(1), object(5)
memory usage: 57.3+ MB
None
date          0
country       0
device        0
continent     0
channel       0
test          0
test_group    0
event_name    0
value         0
dtype: int64


In [7]:
import io


df.columns = ['date', 'country', 'device', 'continent', 'channel', 'test', 'test_group', 'event_name', 'value']
df['date'] = pd.to_datetime(df['date'])

total_columns = df.shape[1]
print(f"1. Загальна кількість колонок: {total_columns}\n")

numeric_columns = df.select_dtypes(include=['number']).columns.tolist()
print(f"2. Кількість числових колонок: {len(numeric_columns)}")
if numeric_columns:
    print(f"   -  Назви: {numeric_columns}\n")

categorical_columns = df.select_dtypes(include=['object', 'category']).columns.tolist()
print(f"3. Кількість категоріальних колонок: {len(categorical_columns)}")
if categorical_columns:
    print(f"   -  Назви: {categorical_columns}\n")

datetime_columns = df.select_dtypes(include=['datetime']).columns.tolist()
print(f"4. Кількість колонок типу datetime: {len(datetime_columns)}")
if datetime_columns:
    print(f"   -  Назви: {datetime_columns}\n")

date_range_start = df['date'].min().strftime('%Y-%m-%d')
date_range_end = df['date'].max().strftime('%Y-%m-%d')
print(f"5. Період часу: з {date_range_start} до {date_range_end}\n")

missing_placeholders = ['(not set)', 'None', '<NA>', '(data deleted)', 'NaN', 'NA']
df_with_na = df.replace(missing_placeholders, pd.NA)
missing_values = df_with_na.isnull().sum()
missing_values = missing_values[missing_values > 0].sort_values(ascending=False)

print("7. Пропущені значення:")

real_missing = df.isnull().sum()
real_missing = real_missing[real_missing > 0]

if real_missing.empty:
    print("   Реальних пропущених значень (NaN) не знайдено.\n")
else:
    print(real_missing)

print("8. Службові / невизначені значення:")

not_set_country = (df['country'] == '(not set)').sum()
not_set_continent = (df['continent'] == '(not set)').sum()

print(f"   country: {not_set_country}")
print(f"   continent: {not_set_continent}")

1. Загальна кількість колонок: 9

2. Кількість числових колонок: 3
   -  Назви: ['test', 'test_group', 'value']

3. Кількість категоріальних колонок: 5
   -  Назви: ['country', 'device', 'continent', 'channel', 'event_name']

4. Кількість колонок типу datetime: 1
   -  Назви: ['date']

5. Період часу: з 2020-11-01 до 2021-01-27

7. Пропущені значення:
   Реальних пропущених значень (NaN) не знайдено.

8. Службові / невизначені значення:
   country: 22063
   continent: 6000


**Перевіряємо і рахуємо події у колонці event_name**

In [8]:
event_counts = df['event_name'].value_counts()
print(event_counts)

event_name
session                107210
session_start          106242
page_view              101907
user_engagement         94520
first_visit             81621
scroll                  73643
view_promotion          61695
view_item               44869
session_with_orders     25892
new account             22389
view_search_results     14310
add_shipping_info       11961
begin_checkout          11959
select_item             11816
add_to_cart             11762
select_promotion         9044
add_payment_info         8048
click                    1997
view_item_list            111
Name: count, dtype: int64


**Створюємо зведену таблицю**

In [9]:
metrics = df["event_name"].unique()
df_pivot = df[df["event_name"].isin(metrics)]
pivot_table = df_pivot.groupby(["test", "test_group", "event_name"])["value"].sum().unstack(fill_value=0).reset_index()
print(pivot_table.to_string())

event_name  test  test_group  add_payment_info  add_shipping_info  add_to_cart  begin_checkout  click  first_visit  new account  page_view  scroll  select_item  select_promotion  session  session_start  session_with_orders  user_engagement  view_item  view_item_list  view_promotion  view_search_results
0              1           1              1988               3034         1395            3784    368        30596         3823     191543   73244          543              1275    45362          45905                 4514           171788      62335              27           29188                 3678
1              1           2              2229               3221         1366            4021    353        30512         3681     198050   73376          530              1323    45193          45649                 4526           179081      65337              24           29117                 3882
2              2           1              2344               3480         2811          

**Тест пропорцій для загальних результатів**

In [10]:
tests = pivot_table['test'].unique()
results = []

for test_id in tests:
    df_test = pivot_table[pivot_table['test'] == test_id]
    group1 =  df_test[df_test['test_group'] == 1]
    group2 =  df_test[df_test['test_group'] == 2]

    for metric in metrics:
        group_1 = group1[metric].sum()
        group_2 = group2[metric].sum()
        div1 = group1['session'].sum()
        div2 = group2['session'].sum()

        conversion1 = group_1 / div1 if div1 > 0 else 0
        conversion2 = group_2 / div2 if div2 > 0 else 0

        if div1 > 0 and div2 > 0 and 0 < group_1 < div1 and 0 < group_2 < div2:
            success = np.array([group_2, group_1])
            total = np.array([div2, div1])
            z_stat, p_value = sm.stats.proportions_ztest(success, total)
        else:
            z_stat, p_value = np.nan, np.nan

        # ((A-B)/B) * 100
        metric_change = (
            ((conversion2 - conversion1) / conversion1) * 100
            if conversion1 > 0
            else np.nan
        )

        significant = (p_value < 0.05) if not np.isnan(p_value) else False

        results.append({
            "test_number": test_id,
            "metric": metric,
            "numerator_event": f"{metric} / session",
            "denominator_event": "session",
            "numerator_1": group_1,
            "denominator_1": div1,
            "control_rate": conversion1,
            "numerator_2": group_2,
            "denominator_2": div2,
            "test_rate": conversion2,
            "metric_change": metric_change,
            "z_stat": z_stat,
            "p_value": p_value,
            "significant": significant}
                       )

In [11]:
results_df = pd.DataFrame(results)
results_df

,test_number,metric,numerator_event,denominator_event,numerator_1,denominator_1,control_rate,numerator_2,denominator_2,test_rate,metric_change,z_stat,p_value,significant
0,1,new account,new account / session,session,3823,45362,0.084278,3681,45193,0.081451,-3.354299,-1.542883,0.122859,False
1,1,session,session / session,session,45362,45362,1.000000,45193,45193,1.000000,0.000000,NaN,NaN,False
2,1,session_with_orders,session_with_orders / session,session,4514,45362,0.099511,4526,45193,0.100148,0.640785,0.320049,0.748931,False
3,1,user_engagement,user_engagement / session,session,171788,45362,3.787046,179081,45193,3.962583,4.635176,NaN,NaN,False
4,1,scroll,scroll / session,session,73244,45362,1.614655,73376,45193,1.623614,0.554845,NaN,NaN,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
71,4,add_payment_info,add_payment_info / session,session,3731,105079,0.035507,3601,105141,0.034249,-3.541234,-1.571106,0.116158,False
72,4,select_promotion,select_promotion / session,session,2773,105079,0.026390,2708,105141,0.025756,-2.401618,-0.911777,0.361886,False
73,4,add_shipping_info,add_shipping_info / session,session,5128,105079,0.048801,4956,105141,0.047137,-3.411125,-1.785795,0.074132,False
74,4,click,click / session,session,285,105079,0.002712,299,105141,0.002844,4.850416,0.572993,0.566649,False


In [12]:
add_payment_info_by_session = results_df[results_df["metric"] == "add_payment_info"]
add_shipping_info_by_session = results_df[results_df["metric"] == "add_shipping_info"]
begin_checkout_by_session = results_df[results_df["metric"] == "begin_checkout"]
new_accounts_by_session = results_df[results_df["metric"] == "new account"]

df_result = pd.concat([add_payment_info_by_session, add_shipping_info_by_session, begin_checkout_by_session, new_accounts_by_session], ignore_index=True)
df_result

,test_number,metric,numerator_event,denominator_event,numerator_1,denominator_1,control_rate,numerator_2,denominator_2,test_rate,metric_change,z_stat,p_value,significant
0,1,add_payment_info,add_payment_info / session,session,1988,45362,0.043825,2229,45193,0.049322,12.542021,3.924884,0.000087,True
1,2,add_payment_info,add_payment_info / session,session,2344,50637,0.046290,2409,50244,0.047946,3.576911,1.240994,0.214608,False
2,3,add_payment_info,add_payment_info / session,session,3623,70047,0.051722,3697,70439,0.052485,1.474630,0.643172,0.520112,False
3,4,add_payment_info,add_payment_info / session,session,3731,105079,0.035507,3601,105141,0.034249,-3.541234,-1.571106,0.116158,False
4,1,add_shipping_info,add_shipping_info / session,session,3034,45362,0.066884,3221,45193,0.071272,6.560481,2.603571,0.009226,True
5,2,add_shipping_info,add_shipping_info / session,session,3480,50637,0.068724,3510,50244,0.069859,1.650995,0.709557,0.477979,False
6,3,add_shipping_info,add_shipping_info / session,session,5298,70047,0.075635,5188,70439,0.073652,-2.621211,-1.413727,0.157442,False
7,4,add_shipping_info,add_shipping_info / session,session,5128,105079,0.048801,4956,105141,0.047137,-3.411125,-1.785795,0.074132,False
8,1,begin_checkout,begin_checkout / session,session,3784,45362,0.083418,4021,45193,0.088974,6.660587,2.978783,0.002894,True
9,2,begin_checkout,begin_checkout / session,session,4262,50637,0.084168,4313,50244,0.085841,1.988164,0.952898,0.340642,False


In [46]:
df_result.to_csv("/content/ab_test_result.csv", index=False)
from google.colab import files
files.download('/content/ab_test_result.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Посилання на [таблицю](https://docs.google.com/spreadsheets/d/15z3MYOBNyDoGlEDHgWtD0uUYVyBsFpLCXIxNNTK3Wss/edit?usp=sharing)

Посилання на [дашборд Tableau](https://public.tableau.com/views/CreateYourABTestingTool_17455624850390/ABTestResultsStatisticalSignificance?:language=en-US&publish=yes&:sid=&:redirect=auth&:display_count=n&:origin=viz_share_link)